# Notebook 03 — Record Store

Builds a versioned covenant record store from Notebook 01's extraction and Notebook 02's confidence scoring, and answers the question the whole project is really about: *what is the governing threshold for a given covenant, as of a given date?*

This matters most for Facility C. Its original agreement states a single, open-ended 3.50:1.00 threshold. Its amendment restates that covenant with two tiers — 3.50:1.00 through Dec 2024, then 4.00:1.00 from Mar 2025 onward. A naive system that only reads the original document, or reads both without resolving which one currently governs, would get every test period from March 2025 onward wrong. This notebook resolves that correctly, by date.

**Input:** `records/01_extraction_output.json`, `records/02_confidence_output.json`
**Output:** `records/03_record_store.json`

**No API calls in this notebook** — everything here is deterministic logic over data you've already extracted and verified.

In [4]:
import json
import re
from datetime import date
from pathlib import Path

from dateutil import parser as dtparser

RECORDS_DIR = Path("../records")

with open(RECORDS_DIR / "01_extraction_output.json") as f:
    extraction_output = json.load(f)

with open(RECORDS_DIR / "02_confidence_output.json") as f:
    confidence_output = json.load(f)

print("Loaded extraction output for", len(extraction_output), "document(s)")
print("Loaded confidence report for", len(confidence_output["confidence_report"]), "document(s)")
print("Items currently in review queue:", len(confidence_output["review_queue"]))

Loaded extraction output for 4 document(s)
Loaded confidence report for 4 document(s)
Items currently in review queue: 1


## Matching extraction to confidence, and filtering to what's actually approved

Notebook 01's extraction and Notebook 02's confidence scoring are two separate files, matched here by citation overlap — the same principle Notebook 02 itself uses, for consistency. Only covenants confirmed `high` confidence are included in the resolvable record store below. Facility B's Reserve Account covenant is currently `low - needs review`, and should be correctly excluded here — not silently treated as reliable just because it was extracted.

In [5]:
DOCUMENT_TO_FACILITY = {
    "facility_a_credit_agreement.txt": "Facility A",
    "facility_b_credit_agreement.txt": "Facility B",
    "facility_c_credit_agreement.txt": "Facility C",
    "facility_c_amendment_1.txt": "Facility C",
}

def citations_overlap(c1, c2):
    return c1 in c2 or c2 in c1

def is_tier_high_confidence(tier, confidence_entries):
    for entry in confidence_entries:
        if entry["confidence"] == "high" and citations_overlap(tier["citation"], entry["primary_citation"]):
            return True
    return False

approved_by_facility = {}
excluded = []

for filename, records in extraction_output.items():
    facility = DOCUMENT_TO_FACILITY[filename]
    confidence_entries = confidence_output["confidence_report"].get(filename, [])
    for record in records:
        approved_tiers = [t for t in record["threshold_tiers"] if is_tier_high_confidence(t, confidence_entries)]
        if approved_tiers:
            approved_record = dict(record, threshold_tiers=approved_tiers, source_document=filename)
            approved_by_facility.setdefault(facility, {}).setdefault(record["covenant_name"], []).append(approved_record)
        for t in record["threshold_tiers"]:
            if t not in approved_tiers:
                excluded.append({"source_document": filename, "covenant_name": record["covenant_name"], "tier": t})

print("Approved covenant records, by facility:")
for facility, covenants in approved_by_facility.items():
    for name, records in covenants.items():
        total_tiers = sum(len(r["threshold_tiers"]) for r in records)
        print(f"  {facility} | {name}: {len(records)} document(s), {total_tiers} approved tier(s)")

print(f"\nExcluded (not high confidence -- unresolved, needs review): {len(excluded)}")
for e in excluded:
    print(f"  {e['source_document']} | {e['covenant_name']}: {e['tier']['threshold_value']}")

Approved covenant records, by facility:
  Facility A | Delinquency Ratio / Delinquency Trigger: 1 document(s), 1 approved tier(s)
  Facility A | Overcollateralization Test: 1 document(s), 1 approved tier(s)
  Facility B | Delinquency Ratio / Delinquency Trigger: 1 document(s), 1 approved tier(s)
  Facility C | Consolidated Net Leverage Ratio: 2 document(s), 3 approved tier(s)

Excluded (not high confidence -- unresolved, needs review): 1
  facility_b_credit_agreement.txt | Reserve Account Deficiency / Required Reserve Amount: 2.00% of Original Pool Balance


## Parsing dates out of extracted text

`applies_from` and `applies_to` aren't always clean ISO dates — extraction returned things like `"Closing Date (March 14, 2024)"` and `"Test Period ending on or after September 30, 2023"`. This needs a parser that can pull a real date out of surrounding descriptive text.

Worth being explicit about a real bug caught while building this: a naive fuzzy date parser on `"Closing Date (March 14, 2024)"` silently returned **2026-03-14** instead of 2024 — the trailing `)` right after the year confused the parser into dropping the year entirely and defaulting to today's year, with no error raised. The fix below checks for a parenthetical date first and parses that in isolation, which resolves it correctly. This is exactly the kind of silent, undetectable-by-eye error this whole project exists to catch — worth stating plainly rather than glossing over.

In [6]:
def parse_flexible_date(s):
    """Parse a date out of a descriptive string. Prefers a parenthetical date if present,
    since fuzzy parsing can otherwise drop the year when a closing paren immediately follows it."""
    if s is None:
        return None
    paren_match = re.search(r'\(([^)]+)\)', s)
    if paren_match:
        try:
            return dtparser.parse(paren_match.group(1), fuzzy=True).date()
        except (ValueError, OverflowError):
            pass  # fall through to parsing the whole string instead
    return dtparser.parse(s, fuzzy=True).date()


# Quick self-check against every real date string extraction has produced so far
_test_cases = {
    "Closing Date (March 14, 2024)": date(2024, 3, 14),
    "Closing Date (November 4, 2024)": date(2024, 11, 4),
    "Test Period ending on or after September 30, 2023": date(2023, 9, 30),
    "September 30, 2023": date(2023, 9, 30),
    "December 31, 2024": date(2024, 12, 31),
    "March 31, 2025": date(2025, 3, 31),
}
for s, expected in _test_cases.items():
    actual = parse_flexible_date(s)
    status = "OK" if actual == expected else f"MISMATCH (expected {expected})"
    print(f"{s!r:55} -> {actual}  [{status}]")

'Closing Date (March 14, 2024)'                         -> 2024-03-14  [OK]
'Closing Date (November 4, 2024)'                       -> 2024-11-04  [OK]
'Test Period ending on or after September 30, 2023'     -> 2023-09-30  [OK]
'September 30, 2023'                                    -> 2023-09-30  [OK]
'December 31, 2024'                                     -> 2024-12-31  [OK]
'March 31, 2025'                                        -> 2025-03-31  [OK]


In [7]:
for name, records in approved_by_facility["Facility C"].items():
    print(f"{name}:")
    for record in records:
        print(f"  from {record['source_document']}  (supersedes={record.get('supersedes')!r})")
        for t in record["threshold_tiers"]:
            print(f"    {t['threshold_value']}  applies_from={t.get('applies_from')!r}  applies_to={t.get('applies_to')!r}")

Consolidated Net Leverage Ratio:
  from facility_c_credit_agreement.txt  (supersedes=None)
    3.50:1.00  applies_from='Test Period ending on or after September 30, 2023'  applies_to=None
  from facility_c_amendment_1.txt  (supersedes='Section 6.01 (Consolidated Net Leverage Ratio) of the Existing Credit Agreement, dated as of June 3, 2023')
    3.50:1.00  applies_from='September 30, 2023'  applies_to='December 31, 2024'
    4.00:1.00  applies_from='March 31, 2025'  applies_to=None


## Merging into a single, priority-ordered timeline

Each covenant may have contributions from more than one document — Facility C's leverage ratio does, with one tier from the original agreement and two from its amendment. The amendment's `supersedes` field tells us it fully restates that covenant ("Section 6.01... is hereby amended and restated in its entirety"), so its tiers take priority over the original's wherever they apply. The original's tier only matters for dates the amendment's tiers don't cover.

This is implemented as a simple priority rule: any tier coming from a record with `supersedes` set is checked first when resolving a date; the original's tier is only reached if nothing higher-priority matches.

In [8]:
def build_timeline(covenant_records):
    """Each record may or may not have 'supersedes' set. Records WITH supersedes
    get higher priority -- they win over an earlier record's tiers wherever their
    own tiers actually provide coverage; the earlier record's tiers only matter
    for dates the superseding record doesn't address."""
    entries = []
    for record in covenant_records:
        priority = 1 if record.get("supersedes") else 0
        for tier in record["threshold_tiers"]:
            entries.append({
                "from": parse_flexible_date(tier.get("applies_from")),
                "to": parse_flexible_date(tier.get("applies_to")),
                "value": tier["threshold_value"],
                "priority": priority,
                "source": record["source_document"],
            })
    # Highest priority first, so resolution checks amendments before originals
    entries.sort(key=lambda e: -e["priority"])
    return entries


# Quick check against Facility C's real approved records
timeline = build_timeline(approved_by_facility["Facility C"]["Consolidated Net Leverage Ratio"])
for e in timeline:
    print(f"  {e['value']}  from={e['from']}  to={e['to']}  priority={e['priority']}  source={e['source']}")

  3.50:1.00  from=2023-09-30  to=2024-12-31  priority=1  source=facility_c_amendment_1.txt
  4.00:1.00  from=2025-03-31  to=None  priority=1  source=facility_c_amendment_1.txt
  3.50:1.00  from=2023-09-30  to=None  priority=0  source=facility_c_credit_agreement.txt


## Resolving a query date

Given a timeline sorted highest-priority-first, resolving a date is simple: walk the list in order and return the first tier whose range contains the queried date. Because amendment tiers are sorted first, they're checked before the original ever gets a chance — the original only wins if no higher-priority tier covers that date at all.

In [9]:
def resolve(timeline, as_of):
    for entry in timeline:
        after_start = entry["from"] is None or as_of >= entry["from"]
        before_end = entry["to"] is None or as_of <= entry["to"]
        if after_start and before_end:
            return entry
    return None


test_dates = {
    date(2023, 8, 1): "before the covenant starts -- should be unresolved",
    date(2023, 11, 30): "Q3 2023 test period -- should be 3.50x",
    date(2024, 12, 31): "Q4 2024 test period, the cured breach -- should be 3.50x",
    date(2025, 3, 31): "Q1 2025 test period, the waived/amended one -- should be 4.00x",
    date(2025, 6, 30): "Q2 2025 -- should be 4.00x",
}

print("Correct resolver (uses both original and amendment):\n")
for d, expectation in test_dates.items():
    result = resolve(timeline, d)
    value = result["value"] if result else "NOT GOVERNED"
    print(f"  {d}: {value}   [{expectation}]")

# What a naive resolver -- one that only ever read the original agreement,
# and never knew the amendment existed -- would have said instead:
naive_timeline = build_timeline(
    [r for r in approved_by_facility["Facility C"]["Consolidated Net Leverage Ratio"]
     if r["source_document"] == "facility_c_credit_agreement.txt"]
)
naive_result = resolve(naive_timeline, date(2025, 6, 30))
print(f"\nNaive resolver (original document only) at 2025-06-30: {naive_result['value']}")
print("This is WRONG -- it misses the amendment entirely and would incorrectly")
print("apply the old 3.50x threshold to a period the amendment actually governs at 4.00x.")

Correct resolver (uses both original and amendment):

  2023-08-01: NOT GOVERNED   [before the covenant starts -- should be unresolved]
  2023-11-30: 3.50:1.00   [Q3 2023 test period -- should be 3.50x]
  2024-12-31: 3.50:1.00   [Q4 2024 test period, the cured breach -- should be 3.50x]
  2025-03-31: 4.00:1.00   [Q1 2025 test period, the waived/amended one -- should be 4.00x]
  2025-06-30: 4.00:1.00   [Q2 2025 -- should be 4.00x]

Naive resolver (original document only) at 2025-06-30: 3.50:1.00
This is WRONG -- it misses the amendment entirely and would incorrectly
apply the old 3.50x threshold to a period the amendment actually governs at 4.00x.


## Building and saving the full record store

Every facility and covenant that survived Notebook 02's confidence filter gets its own resolved timeline here — not just Facility C. Reserve Account is deliberately absent: it never made it past Cell 4's filter, and its absence from this store is itself the correct, intended behavior — an unresolved covenant should be unavailable to the resolver, not silently included.

In [10]:
record_store = {}
for facility, covenants in approved_by_facility.items():
    record_store[facility] = {}
    for covenant_name, records in covenants.items():
        timeline = build_timeline(records)
        record_store[facility][covenant_name] = [
            {
                "value": e["value"],
                "from": e["from"].isoformat() if e["from"] else None,
                "to": e["to"].isoformat() if e["to"] else None,
                "priority": e["priority"],
                "source": e["source"],
            }
            for e in timeline
        ]

output_path = RECORDS_DIR / "03_record_store.json"
output_path.write_text(json.dumps(record_store, indent=2))

print(f"Record store saved to {output_path}\n")
for facility, covenants in record_store.items():
    for name, timeline in covenants.items():
        print(f"  {facility} | {name}: {len(timeline)} tier(s)")

print(f"\nExcluded from the store (unresolved, needs review): {len(excluded)}")
for e in excluded:
    print(f"  {e['source_document']} | {e['covenant_name']}: {e['tier']['threshold_value']}")

Record store saved to ../records/03_record_store.json

  Facility A | Delinquency Ratio / Delinquency Trigger: 1 tier(s)
  Facility A | Overcollateralization Test: 1 tier(s)
  Facility B | Delinquency Ratio / Delinquency Trigger: 1 tier(s)
  Facility C | Consolidated Net Leverage Ratio: 3 tier(s)

Excluded from the store (unresolved, needs review): 1
  facility_b_credit_agreement.txt | Reserve Account Deficiency / Required Reserve Amount: 2.00% of Original Pool Balance


## Summary

Reloads the saved record store from disk and resolves a handful of real dates across every facility — not just Facility C — to confirm the whole store works end to end, not only the one case we stress-tested most.

In [11]:
with open(RECORDS_DIR / "03_record_store.json") as f:
    saved_store = json.load(f)

def deserialize_timeline(entries):
    result = []
    for e in entries:
        result.append({
            "value": e["value"],
            "from": date.fromisoformat(e["from"]) if e["from"] else None,
            "to": date.fromisoformat(e["to"]) if e["to"] else None,
            "priority": e["priority"],
            "source": e["source"],
        })
    return result

print("=== Record store contents ===")
for facility, covenants in saved_store.items():
    print(f"\n{facility}:")
    for name in covenants:
        print(f"  - {name}")

print("\n=== Illustrative resolutions ===\n")

checks = [
    ("Facility A", "Delinquency Ratio / Delinquency Trigger", date(2024, 6, 30)),
    ("Facility A", "Overcollateralization Test", date(2024, 10, 31)),
    ("Facility B", "Delinquency Ratio / Delinquency Trigger", date(2025, 3, 31)),
    ("Facility C", "Consolidated Net Leverage Ratio", date(2023, 11, 30)),
    ("Facility C", "Consolidated Net Leverage Ratio", date(2025, 6, 30)),
]

for facility, covenant, as_of in checks:
    timeline = deserialize_timeline(saved_store[facility][covenant])
    result = resolve(timeline, as_of)
    value = result["value"] if result else "NOT GOVERNED"
    print(f"  {facility} | {covenant} | as of {as_of}: {value}")

print("\n=== Deliberately unavailable ===")
print("  Facility B | Reserve Account Deficiency: not in the store (still needs human review)")

=== Record store contents ===

Facility A:
  - Delinquency Ratio / Delinquency Trigger
  - Overcollateralization Test

Facility B:
  - Delinquency Ratio / Delinquency Trigger

Facility C:
  - Consolidated Net Leverage Ratio

=== Illustrative resolutions ===

  Facility A | Delinquency Ratio / Delinquency Trigger | as of 2024-06-30: 5.00%
  Facility A | Overcollateralization Test | as of 2024-10-31: 8.00%
  Facility B | Delinquency Ratio / Delinquency Trigger | as of 2025-03-31: 4.00%
  Facility C | Consolidated Net Leverage Ratio | as of 2023-11-30: 3.50:1.00
  Facility C | Consolidated Net Leverage Ratio | as of 2025-06-30: 4.00:1.00

=== Deliberately unavailable ===
  Facility B | Reserve Account Deficiency: not in the store (still needs human review)
